# Week 4 Day 3 - Exercise: browser agent with memory and a block list

Exercise only. It reuses the Playwright MCP browser tools from Part 6 of the lab, which need Node v22+ (`npx` fetches Playwright on demand).

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(override=True)

In [ ]:
client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Give the browser agent a checkpointer and a thread_id, then hold a short back and forth with it: ask it to open a site, then ask a follow up question about what it found, and confirm it remembers. For a bigger challenge, write a piece of middleware that refuses any navigation to a site you put on a block list, and prove it works.
            </span>
        </td>
    </tr>
</table>

## Exercise, step 1: a browser agent with memory

Same `browser_tools` as before, but now with a `MemorySaver` checkpointer and a `thread_id`, so a follow-up question can rely on what the agent already saw.

In [ ]:
browser_agent_mem = create_agent(
    model="openai:gpt-5.4-mini",
    tools=browser_tools,
    checkpointer=MemorySaver(),
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
)

mem_config = {"configurable": {"thread_id": "browsing"}}

result = await browser_agent_mem.ainvoke(
    {"messages": [{"role": "user", "content": "Go to https://news.ycombinator.com and tell me the titles of the top three stories."}]},
    config=mem_config,
)
print(result["messages"][-1].content)

In [ ]:
# Follow-up: no site named, so it only works if the agent remembers the conversation
result = await browser_agent_mem.ainvoke(
    {"messages": [{"role": "user", "content": "What was the title of the second story you just told me about?"}]},
    config=mem_config,
)
print(result["messages"][-1].content)

## Exercise, step 2: block list middleware

Middleware that wraps every tool call. If the tool is `browser_navigate` and the URL's domain is on the block list, we do not call `handler`; we return a `ToolMessage` saying the site is blocked, so the agent can tell the user why.

In [ ]:
from urllib.parse import urlparse
from langchain_core.messages import ToolMessage

BLOCKED_SITES = ["reddit.com"]

def is_blocked(url: str) -> bool:
    host = (urlparse(url).hostname or "").lower()
    return any(host == site or host.endswith("." + site) for site in BLOCKED_SITES)

@wrap_tool_call
async def block_sites(request, handler):
    call = request.tool_call
    if call["name"] == "browser_navigate" and is_blocked(call["args"].get("url", "")):
        print(f"  [middleware] BLOCKED navigation to {call['args']['url']}")
        return ToolMessage(
            content=f"Navigation refused: {call['args']['url']} is on the block list. Do not try to reach it another way.",
            tool_call_id=call["id"],
        )
    return await handler(request)

guarded_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=browser_tools,
    middleware=[block_sites],
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
)

In [ ]:
# Should be blocked
result = await guarded_agent.ainvoke({"messages": [{"role": "user", "content": "Go to https://www.reddit.com and tell me the top post."}]})
print(result["messages"][-1].content)

In [ ]:
# Should still work
result = await guarded_agent.ainvoke({"messages": [{"role": "user", "content": "Go to https://news.ycombinator.com and tell me the title of the top story."}]})
print(result["messages"][-1].content)